In [2]:
 # 引入sqlite 轻量数据库
import sqlite3

In [3]:
# 打开链接
# 打开或创建一个数据库文件
conn=sqlite3.connect("test.db")
# 数据库操作的句柄
# 用于执行SQL语句，获取查询结果
cursor=conn.cursor()
# 创建表
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees(
    id INTEGER PRIMARY KEY,
    name TEXT,
    departerment TEXT,
    salary INTEGER

)
""")

In [ ]:
# 添加数据
sample_data=[
    (6,"陈昊","开发部",32000),
    (7,"张三","销售部",20000),
    (8,"曹伟伟","开发部",33000),
    (9,"李四","销售部",15000)
]
cursor.executemany('INSERT INTO employees VALUES(?,?,?,?)',sample_data)
# 提交事务，确保数据真正写入数据库 否则不会保存
conn.commit()

In [5]:
from openai import OpenAI
client=OpenAI(
    api_key="sk-a4b8a559bf374177a7faf38608bafcff",
    base_url="https://api.deepseek.com"
)

In [6]:
# 通过table_info 拿到employees 表的描述
# llm 生成sql 提供上下文
schema=cursor.execute("PRAGMA table_info(employees)").fetchall()
#  f""模板字符串  col->从0开始
schema_str = "CREATE TABLE EMPLOYEES (\n" + "\n".join([f"{col[1]} {col[2]}" for col in schema]) + "\n)"
print("数据库Schema:")
print(schema_str)

数据库Schema:
CREATE TABLE EMPLOYEES (
id INTEGER
name TEXT
departerment TEXT
salary INTEGER
)


In [7]:
def ask_deepseek(query, schema):
    prompt = f"""
    这是一个数据库的Schema:
    {schema}
    根据这个Schema，你能输出一个SQL查询，来回答以下问题吗？
    只输出SQL查询语句，不要输出任何其他内容，也不要带任何格式。
    问题：{query}
    """
    response = client.chat.completions.create(
        model='deepseek-reasoner',
        max_tokens=2048,
        messages=[
            {'role': 'user', 'content': prompt}
        ]
    )
    return response.choices[0].message.content.strip()

# ask_deepseek("开发部部门员工的姓名和工资是多少",schema)
question='开发部部门员工的工资是多少?'
sql=ask_deepseek(question,schema_str)
print('生成的sql查询:')
print(sql)

生成的sql查询:
SELECT salary FROM EMPLOYEES WHERE departerment = '开发部';
